In [1]:
import datetime as dt

from moexalgo import CandlePeriod

import luigi
from auction_model import AuctionModelInference, TargetType, CalibFrequency
from graph import DEFAULT_BASE_DIR as BASE_DIR, InMemoryTask, DailyMixin
from qr import get_dates_range
from tasks import Board, BondType, BondsDescription, BondMetricsAtVWAP, YieldCandles, AuctionDates, \
    AuctionResultsHistory

luigi.interface.core.log_level = "WARNING"

In [2]:
from tasks import Candles

class ADV(InMemoryTask, DailyMixin):

    def requires(self):
        return [self.clone(Candles, date=date, board=Board.TQOB, period=CandlePeriod.ONE_HOUR) for date in get_dates_range(self.prev_date, 10)]

    def produce_output(self):
        value_df = pd.concat(t.read_output()['volume'].unstack().T.between_time(dt.time(10), dt.time(19)) for t in self.requires())
        return value_df.groupby(value_df.index.to_series().dt.date).sum().mean().to_frame('adv') * 1e3

In [208]:
luigi.build([AuctionDates(base_dir=BASE_DIR), AuctionResultsHistory(base_dir=BASE_DIR)], local_scheduler=True)
auction_dates = AuctionDates(base_dir=BASE_DIR).read_output().query('20250101 < announce_date < 20251231')

In [1361]:
import pandas as pd
from qr import shift_mos_date

SHIFTS = [(-2, 10), (-2, 12), (-2, 14), (-2, 18), (-1, 10), (-1, 14), (-1, 18), (0, 10), (0, 14), (0, 18), (1, 10), (1, 18), (2, 18), (3, 18)]
# SHIFTS = [(days_shift, hour) for days_shift in [-2, -1, 0] for hour in range(10, 20)]
dfs = []

def shift_col_name(shift_bdays, hour):
    shift_bdays_map = {-3: 'THU-1', -2: 'FRI-1', -1: 'MON', 0: 'TUE', 1: 'WED', 2: 'THU', 3: 'FRI'}
    return f"{shift_bdays_map[shift_bdays]}_{hour}"

for _, row in auction_dates.iterrows():
    date = row['announce_date'].date()
    descr = BondsDescription(date=date, base_dir=BASE_DIR)
    metrics = descr.clone(BondMetricsAtVWAP, bond_type=BondType.FIX)
    inference = descr.clone_previous(AuctionModelInference, time_of_day=dt.timedelta(hours=10),
                            calib_frequency=CalibFrequency.QUARTERLY, target_type=TargetType.FIRST,
                            loss_function='QuerySoftMax', eval_metric='QuerySoftMax')
    tasks = [descr, metrics, inference]
    luigi.build(tasks, local_scheduler=True)

    df = metrics.read_output()[['duration']].join(descr.read_output()[['series']]).join(inference.read_output().reset_index().set_index('secid')[['score', 'target']])

    for shift_bdays, hour in SHIFTS:
        shift_date = shift_mos_date(date, shift_bdays)
        timestamp = pd.to_datetime(shift_date) + pd.Timedelta(hours=hour)
        candles = descr.clone(YieldCandles, date=shift_date, bond_type=BondType.FIX, board=Board.TQOB, period=CandlePeriod.ONE_HOUR)
        luigi.build([candles], local_scheduler=True)
        df[shift_col_name(shift_bdays, hour)] = candles.read_output().xs(key=timestamp, level='begin')['weighted_avg']

    dfs.append(df.assign(announce_date=pd.to_datetime(date)))

In [1362]:
df = pd.concat(dfs)
df = df.reset_index().merge(AuctionResultsHistory(base_dir=BASE_DIR).read_output()[['series', 'announce_date', 'demand_mln', 'average_yield', 'cutoff_yield']].rename(columns={'average_yield':'auction_average_yield', 'cutoff_yield': 'auction_cutoff_yield'}), on=['series', 'announce_date'], how='left', indicator='euler_target')
df['target'] = df['euler_target'].map({'left_only': 0, 'both': 1})

In [1363]:
df = df.reset_index().set_index(['announce_date', 'secid'], verify_integrity=True).reset_index()

In [1364]:
df = df[df['duration'] > 3].copy()
df['target1_count'] = df.groupby('announce_date')['target'].transform('sum')

In [1365]:
for shift_bdays, hour in SHIFTS:
    col_name = shift_col_name(shift_bdays, hour)
    df[f'raw_delta_{col_name}'] = df[col_name] - df[shift_col_name(*SHIFTS[0])]
    df[f'demean_delta_{col_name}'] = df[f'raw_delta_{col_name}'] - df.groupby('announce_date')[f'raw_delta_{col_name}'].transform('median')

In [1366]:
# df[f'raw_delta_auc_avg'] = df['auction_average_yield'] - df[shift_col_name(*SHIFTS[0])]
# df[f'raw_delta_auc_cutoff'] = df['auction_cutoff_yield'] - df[shift_col_name(*SHIFTS[0])]

In [1367]:
df['score_rank'] = df.sort_values('duration', ascending=False).groupby('announce_date')['score'].rank(ascending=False, method='first')
assert not df[['announce_date', 'score_rank']].dropna().duplicated().any()
def map_rank(x):
    if pd.isna(x):
        return -1
    elif x <= 3:
        return 1
    else:
        return 0
df['score_group'] = df['score_rank'].map(map_rank)

ddf = df.groupby(['score_group', 'target'])[df.filter(like='raw_delta_').columns].agg(['count', 'mean', 'median', 'std', 'min', 'max']).T.round(2)

In [1390]:
df.groupby(['score_group', 'announce_date'])[df.filter(like='raw_delta_').columns].agg(['mean', 'std', 'count']).swaplevel(axis=1)['count']

#.join(df.groupby(['score_group', 'announce_date']).size().to_frame('size'))

raw_delta_FRI-1_10  raw_delta_FRI-1_12  \
score_group announce_date                                           
-1          2025-01-14                     17                  17   
            2025-01-21                     16                  16   
            2025-01-28                     16                  16   
            2025-02-04                     10                  10   
            2025-02-11                      6                   6   
...                                       ...                 ...   
 1          2025-11-25                      3                   3   
            2025-12-02                      3                   3   
            2025-12-09                      3                   3   
            2025-12-16                      3                   3   
            2025-12-23                      3                   3   

                           raw_delta_FRI-1_14  raw_delta_FRI-1_18  \
score_group announce_date                                           
-1          2025-01-14                     17                  17   
            2025-01-21                     16                  16   
            2025-01-28                     16                  16   
            2025-02-04                     10                  10   
            2025-02-11                      6                   6   
...                                       ...                 ...   
 1          2025-11-25                      3                   3   
            2025-12-02                      3                   3   
            2025-12-09                      3                   3   
            2025-12-16                      3                   3   
            2025-12-23                      3                   3   

                           raw_delta_MON_10  raw_delta_MON_14  \
score_group announce_date                                       
-1          2025-01-14                   17                17   
            2025-01-21                   16                16   
            2025-01-28                   16                16   
            2025-02-04                   10                10   
            2025-02-11                    6                 6   
...                                     ...               ...   
 1          2025-11-25                    3                 3   
            2025-12-02                    3                 3   
            2025-12-09                    3                 3   
            2025-12-16                    3                 3   
            2025-12-23                    3                 3   

                           raw_delta_MON_18  raw_delta_TUE_10  \
score_group announce_date                                       
-1          2025-01-14                   17                17   
            2025-01-21                   16                16   
            2025-01-28                   16                16   
            2025-02-04                   10                10   
            2025-02-11                    6                 6   
...                                     ...               ...   
 1          2025-11-25                    3                 3   
            2025-12-02                    3                 3   
            2025-12-09                    3                 3   
            2025-12-16                    3                 3   
            2025-12-23                    3                 3   

                           raw_delta_TUE_14  raw_delta_TUE_18  \
score_group announce_date                                       
-1          2025-01-14                   17                17   
            2025-01-21                   16                16   
            2025-01-28                   16                16   
            2025-02-04                   10                10   
            2025-02-11                    6                 6   
...                                     ...               ...   
 1          2025-11

In [1368]:
# display(df[df.groupby('announce_date')['score'].transform('count') >= 10][df.filter(like='raw_delta_').columns].mean().T.round(2))
display(df[df.groupby('announce_date')['score'].transform('count') >= 10].groupby(['score_group'])[df.filter(like='raw_delta_').columns].mean().T.round(2))
display(df[df.groupby('announce_date')['score'].transform('count') >= 10].groupby(['target'])[df.filter(like='raw_delta_').columns].mean().T.round(2))
display(df.assign(score_notna=df['score'].notna())[df.groupby('announce_date')['score'].transform('count') >= 10].groupby(['score_notna', 'target'])[df.filter(like='raw_delta_').columns].mean().T.round(2))

score_group,-1,0,1
raw_delta_FRI-1_10,0.00,0.00,0.00
raw_delta_FRI-1_12,0.00,0.00,0.01
raw_delta_FRI-1_14,0.03,0.03,0.04
raw_delta_FRI-1_18,0.03,0.04,0.04
raw_delta_MON_10,0.02,0.02,0.01
raw_delta_MON_14,0.05,0.06,0.06
raw_delta_MON_18,0.05,0.06,0.06
raw_delta_TUE_10,0.03,0.03,0.03
raw_delta_TUE_14,0.04,0.05,0.06
raw_delta_TUE_18,0.04,0.06,0.08


target,0.0,1.0
raw_delta_FRI-1_10,0.00,0.00
raw_delta_FRI-1_12,0.00,0.01
raw_delta_FRI-1_14,0.03,0.04
raw_delta_FRI-1_18,0.04,0.04
raw_delta_MON_10,0.02,0.01
raw_delta_MON_14,0.06,0.05
raw_delta_MON_18,0.06,0.05
raw_delta_TUE_10,0.03,0.02
raw_delta_TUE_14,0.05,0.05
raw_delta_TUE_18,0.05,0.11


score_notna        False True       
target               0.0   0.0   1.0
raw_delta_FRI-1_10  0.00  0.00  0.00
raw_delta_FRI-1_12  0.00  0.01  0.01
raw_delta_FRI-1_14  0.03  0.03  0.04
raw_delta_FRI-1_18  0.03  0.04  0.04
raw_delta_MON_10    0.02  0.02  0.01
raw_delta_MON_14    0.05  0.06  0.05
raw_delta_MON_18    0.05  0.06  0.05
raw_delta_TUE_10    0.03  0.03  0.02
raw_delta_TUE_14    0.04  0.05  0.05
raw_delta_TUE_18    0.04  0.05  0.11
raw_delta_WED_10    0.04  0.05  0.12
raw_delta_WED_18    0.03  0.04  0.08
raw_delta_THU_18   -0.01 -0.01  0.03
raw_delta_FRI_18   -0.01  0.01  0.03

In [1258]:
from enum import Enum

class Side(Enum):
    BUY = 1
    SELL = -1

    def worsen_price(self, price, offset):
        return price + offset * self.value

In [1419]:
import pandas as pd
from collections import defaultdict
from calendar import Day
import numpy as np


class Portfolio:

    def __init__(self):
        self.positions = defaultdict(float)
        self.cash = 0.0
        self.trades = []

    def get_position(self, secid):
        return self.positions.get(secid, 0.0)

    def update_trade(self, date, secid, qty, price, source):
        self.positions[secid] += qty
        value = qty * price / 100
        self.cash -= value

        self.trades.append({
            "date": date,
            "secid": secid,
            "qty": qty,
            "price": price,
            "value": value,
            "source": source
        })

    def market_value(self, price_map):
        mv = self.cash
        for secid, qty in self.positions.items():
            mv += qty * price_map[secid] / 100

        return mv


class ExecutionModel:

    def __init__(self, cost_model):
        self.cost_model = cost_model
        self.auc_data = AuctionResultsHistory(base_dir=BASE_DIR).read_output()[['date', 'series', 'average_price']].dropna()
        self.auc_data['date'] = self.auc_data['date'].dt.date

    def execute(self, date: dt.date, qty, side, vwap_price, bond_info):

        cost_bps = self.cost_model.cost_bps(
            bond_info["years_to_maturity"],
            bond_info["adv"],
            qty
        )

        if side == Side.BUY:
            # buy on auction! should be at 15:00:00
            auc_result = self.auc_data[(self.auc_data['date'] == date) & (self.auc_data['series'] == bond_info['series'])]
            if not auc_result.empty:
                return auc_result['average_price'].iloc[0], 'auction'


        cost_price = vwap_price * bond_info['modified_duration'] * cost_bps * 1e-4

        return side.worsen_price(vwap_price, cost_price), 'order_book'



class AuctionStrategy:

    def __init__(self, short_only, action_by_date, short_notionals):
        self.short_only = short_only
        self.action_by_date = action_by_date
        self.short_notionals = short_notionals


    def build_target(self, date, universe_df):
        action = self.action_by_date.get(date.weekday())
        if action is None:
            return None
        target = {}

        inference = AuctionModelInference(date=date, base_dir=BASE_DIR, time_of_day=dt.timedelta(hours=9),
                                          calib_frequency=CalibFrequency.QUARTERLY,
                                          target_type=TargetType.FIRST, loss_function='QuerySoftMax',
                                          eval_metric='QuerySoftMax')
        luigi.build([inference], local_scheduler=True)
        inference_df = inference.read_output().set_index('secid')

        if action == 'model_target':
            inf_sorted = inference_df.sort_values(["score", 'years_to_maturity'], ascending=[False, False])
            top = inf_sorted.head(len(self.short_notionals)).index.tolist()
        elif action == 'true_target':
            top = inference_df.query('target == 1').index.tolist()
        else:
            top = []
            assert action == 'close'

        for secid, size in zip(top, self.short_notionals):
                if secid not in universe_df.index:
                    # print(f'ignoring {secid} as not in universe')
                    pass
                elif np.isnan(universe_df.loc[secid, 'dv01']):
                    # print(f'ignoring {secid} as no dv01')
                    pass
                else:
                    target[secid] = -size


        total_dv01_short = self.calc_total_dv01(target, universe_df)

        # LONGS
        if not self.short_only and len(target) > 0:
            on_run = set(inference_df.index)
            off_run_df = universe_df[~universe_df.index.isin(on_run)].copy()
            off_run_df['ad_dv01'] = off_run_df['adv'] * off_run_df['dv01']
            off_run_df = off_run_df.sort_values("ad_dv01", ascending=False)

            hedge_candidates = off_run_df.head(5).copy()

            hedge_candidates['size'] = -total_dv01_short / hedge_candidates['ad_dv01'].sum() * hedge_candidates['adv']
            for secid, row in hedge_candidates.iterrows():
                target[secid] = row["size"]

            net_dv01 = abs(self.calc_total_dv01(target, universe_df))
            assert net_dv01 < 1e-5

        return target

    def calc_total_dv01(self, target, universe_df):
        total_dv01_short = 0.0
        for secid, size in target.items():
            row = universe_df.loc[secid]
            dv01 = row["dv01"]
            assert not np.isnan(dv01)
            total_dv01_short += size * dv01
        return total_dv01_short


class AuctionBacktester:

    def __init__(self, strategy, execution_model):

        self.strategy = strategy
        self.portfolio = Portfolio()
        self.execution = execution_model

        self.history = []

    def compute_vwap(self, candles_df):
        df = candles_df.copy()
        df["pxv"] = df["weighted_avg"] * df["volume"]
        vwap = df.groupby("secid")[['pxv', 'volume']].sum()
        return (vwap['pxv'] / vwap['volume']).rename('vwap')


    def rebalance(self, date, target, universe_df, vwap):

        current_positions = dict(self.portfolio.positions)

        all_bonds = set(current_positions.keys()).union(target.keys())

        for secid in all_bonds:

            current_qty = current_positions.get(secid, 0.0)
            target_qty = target.get(secid, 0.0)

            delta = target_qty - current_qty

            if abs(delta) < 1e-6:
                continue


            vwap_price = vwap.loc[secid]
            bond_row = universe_df.loc[secid]

            exec_price, source = self.execution.execute(
                date,
                abs(delta),
                Side.BUY if delta > 0 else Side.SELL,
                vwap_price,
                bond_row
            )

            self.portfolio.update_trade(
                date,
                secid,
                delta,
                exec_price,
                source
            )


    def compute_risk(self, universe_df):

        net_dv01 = 0.0
        longs_dv01 = 0.0
        shorts_dv01 = 0.0
        gross_notional = 0.0
        longs_notional = 0.0
        shorts_notional = 0.0

        for secid, qty in self.portfolio.positions.items():

            row = universe_df.loc[secid]
            dv01 = row["dv01"]

            net_dv01 += qty * dv01
            longs_dv01 += max(qty,0) * dv01
            shorts_dv01 += min(qty, 0) * dv01
            gross_notional += abs(qty)
            longs_notional += max(qty, 0)
            shorts_notional += min(qty, 0)


        return net_dv01, longs_dv01, shorts_dv01, gross_notional, longs_notional, shorts_notional

    def calc_metrics(self, date, close_prices, universe_df):

        equity = self.portfolio.market_value(close_prices.to_dict())

        net_dv01, longs_dv01, shorts_dv01, gross_notional, longs_notional, shorts_notional =  self.compute_risk(universe_df)

        self.history.append({
            "date": date,
            "equity": equity,
            "net_dv01": net_dv01,
            "longs_dv01": longs_dv01,
            "shorts_dv01": shorts_dv01,
            "gross_notional": gross_notional,
            "longs_notional": longs_notional,
            "shorts_notional": shorts_notional
        })

    def run(self, dates):
        luigi.build([AuctionResultsHistory(base_dir=BASE_DIR)], local_scheduler=True)

        for date in dates:
            tasks = [BondsDescription(base_dir=BASE_DIR, date=date),
                     Candles(base_dir=BASE_DIR, date=date, period=CandlePeriod.ONE_HOUR, board=Board.TQOB),
                     BondMetricsAtVWAP(base_dir=BASE_DIR, date=date, bond_type=BondType.FIX),
                     ADV(base_dir=BASE_DIR, date=date)]
            luigi.build(tasks, local_scheduler=True)
            universe_df, candles_df, metrics_df, adv_df = (t.read_output() for t in tasks)
            candles_df = candles_df[candles_df['end'].dt.time.between(dt.time(10), dt.time(19))].copy()
            universe_df = universe_df[universe_df['SHORT_TYPE'] == BondType.FIX.value].copy()
            universe_df['years_to_maturity'] = (universe_df['MATDATE'] - pd.to_datetime(date)) / pd.Timedelta(days=365)

            metrics_df['dv01'] = (metrics_df['price'] / 100) * metrics_df['modified_duration'] * 1e-4
            universe_df = universe_df.join(metrics_df[['modified_duration', 'dv01']])
            universe_df = universe_df.join(adv_df)

            vwap = self.compute_vwap(candles_df)

            target = self.strategy.build_target(date, universe_df)
            if target is not None:
                self.rebalance(date, target, universe_df, vwap)

            close_prices = candles_df.reset_index().groupby('secid')['weighted_avg'].last()


            self.calc_metrics(
                date,
                close_prices,
                universe_df
            )

        return pd.DataFrame(self.history)


In [1420]:
class CostModel:

    def cost_bps(self, years_to_maturity, adv, qty):
        if years_to_maturity < 10:
            if qty >= 500_000:
                return 4.
            elif qty >= 350_000:
                return 3.
            else:
                return 2.
        else:
            if qty >= 500_000:
                return 3.
            elif qty >= 350_000:
                return 2.
            else:
                return 2.

class HalfCostModel:

    def cost_bps(self, years_to_maturity, adv, qty):
        return CostModel().cost_bps(years_to_maturity, adv, qty) * 0.5

class ZeroCostModel:

    def cost_bps(self, years_to_maturity, adv, qty):
        return 0.

In [1421]:
from qr import get_mos_dates

dates = get_mos_dates(dt.date(2025, 1, 1), dt.date(2026, 2, 18), incl_end=True)
backtester = AuctionBacktester(AuctionStrategy(short_only=False, action_by_date={Day.WEDNESDAY: 'model_target'}, short_notionals=[500_000_000, 350_000_000, 150_000_000]), ExecutionModel(CostModel()))
history = backtester.run(dates)

In [1414]:
history.set_index('date')['shorts_dv01'].plot(backend='plotly')